# Fine-tune Whisper on CDLI Non-Standard Speech Datasets



In [1]:
from huggingface_hub import login
HF_TOKEN = input()
login(token=HF_TOKEN)

## Settings 

--> adapt for your scenario

### Directories

In [2]:
import os 

# storage in Volume that will persist
LOCAL_STORAGE_DIR = '/jupyter_kernel'

BASE_DIR = os.path.join(LOCAL_STORAGE_DIR, 'trained_models')
!mkdir -p {BASE_DIR}

# directory for model training
OUTPUT_DIR = os.path.join(BASE_DIR, 'en_nonstandard_tune_whisper_large_v3_baseline')
#OUTPUT_DIR = os.path.join(BASE_DIR, 'sw_nonstandard_tune_whisper_large_V3_1')

print(f"Will write model to: {OUTPUT_DIR}")
if os.path.exists(OUTPUT_DIR):
    raise ValueError(f"Output directory already exists - if you continue this will overwrite data and may lead to strange results...")


Will write model to: /jupyter_kernel/trained_models/en_nonstandard_tune_whisper_large_v3_baseline


ValueError: Output directory already exists - if you continue this will overwrite data and may lead to strange results...

### Model and Dataset settings

In [3]:
# WHISPER_MODEL_TYPE = "openai/whisper-tiny" 
# WHISPER_MODEL_TYPE = "openai/whisper-small" 
WHISPER_MODEL_TYPE = "openai/whisper-large-v3" 

LANGUAGE = 'en'
DATASET_NAME = "cdli/kenyan_english_nonstandard_speech_v0.9"

# LANGUAGE = 'sw'
#DATASET_NAME = "cdli/kenyan_swahili_nonstandard_speech_v0.9"


In [4]:
# which parts of the model to update
UPDATE_ENCODER = True
UPDATE_PROJ = True
UPDATE_DECODER = False

# Turn on SpecAugment
USE_SPECAUGMENT = True

In [5]:

#######################
## don't change these!
######################


TASK = "transcribe"

BASE_MODEL_NAME = WHISPER_MODEL_TYPE
print('Base model will be loaded from:', BASE_MODEL_NAME)

Base model will be loaded from: openai/whisper-large-v3


### Trainer Settings

--> adjust as needed or keep defaults (these settings should be a good starting point)

In [6]:

LOGGING_STEPS = 10
# if save steps is 0, only last and best model will be written
SAVE_STEPS = 100

# training duration
MAX_EPOCHS = 10
MAX_STEPS = 1000  # for larger datasets, you will want to increase this

# Learning Rate and LR Scheduler (LR_END and LR_DECAY_POWER only apply to polynomial)
LEARNING_RATE = 3e-5 #@param
LR_SCHEDULER_TYPE = 'polynomial' # constant_with_warmup or polynomial
LR_WARMUP_STEPS = 100
LR_END = 1e-7
LR_DECAY_POWER = 2
# see: https://huggingface.co/docs/transformers/v4.46.2/en/main_classes/optimizer_schedules#transformers.SchedulerType
# and here: https://www.kaggle.com/code/snnclsr/learning-rate-schedulers
# constant --> 'constant_with_warmup'
# polynomial --> 'get_polynomial_decay_schedule_with_warmup'

BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8

#@markdown other settings relevant for evaluation
MAX_GEN_LEN = 224 # increase if your data has long sequences!
EVAL_ON_START = True
EVAL_STEPS = 100

# for CPU, set both to false
USE_FP16 = True
USE_BF16 = False # only some GPUs support this, eg A100, A40

# checkpoints get huge for large models (~18 GB!)
NUM_CHECKPOINTS_TO_STORE = 2

## Imports and Prep

--> no need to change anything here, just run

In [7]:
import datasets
from huggingface_hub import hf_hub_download
import numpy as np
import pandas as pd
import os
import torch

# more efficient dataset handling
datasets.disable_caching()
print('cache:', datasets.is_caching_enabled())

torch.set_num_threads(1)
torch.get_num_threads()

# check if we have gpu
if torch.cuda.is_available():
    print("GPU is available")
else:
    print("GPU is not available, using CPU instead")

cache: False
GPU is available


In [9]:
from huggingface_hub import hf_hub_download

import random
import torchaudio
import librosa


import tarfile
import datasets
import matplotlib.pyplot as plt
import pandas as pd

import torch
import time


from dataclasses import dataclass
from typing import Any, Dict, List, Union

from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer

from transformers import WhisperProcessor
from transformers import WhisperForConditionalGeneration
import os
import csv
import shutil
import numpy as np


import evaluate
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

transcript_normalizer = BasicTextNormalizer()

In [10]:
def count_trainable_parameters(model):
    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    return params

In [11]:
def get_wer(references, predictions, normalize=True, verbose=True):
  rs = references
  ps = predictions
  if normalize:
    ps = [transcript_normalizer(x) for x in predictions]
    rs = [transcript_normalizer(x) for x in references]
  if verbose:
    for r, p in zip(rs, ps):
      print(r)
      print(p)
      print()

  return wer_metric.compute(references=rs, predictions=ps)


def compute_metrics(pred):
    # for training metrics
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_strs = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_strs = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # calculate a per-example average
    wers = []
    cers = []
    for pred_str, label_str in zip(pred_strs, label_strs):
      p = transcript_normalizer(pred_str)
      l = transcript_normalizer(label_str)
      wer = wer_metric.compute(predictions=[p], references=[l])
      cer = cer_metric.compute(predictions=[p], references=[l])
      wers.append(wer)
      cers.append(cer)

    wer = np.mean([min(1.0,x) for x in wers])
    cer = np.mean([min(1.0,x) for x in cers])
    print('adjusted:', wer, cer)
    print('un-adjusted:', np.mean(wers), np.mean(cers))
    return {"wer": wer, "cer": cer}



In [12]:
def load_dataset(dataset_name, split='test', limit_to_30_seconds=True):
    """
    Load a dataset from Hugging Face Hub.
    If limit_to_30_seconds is True, will only load examples with audio length <= 30 seconds.
    """
    if split not in ['train', 'test', 'validation']:
        raise ValueError("split must be one of 'train', 'test', or 'validation'")
    ds = datasets.load_dataset(dataset_name, split=split, streaming=False)
    orig_len = len(ds)
    if limit_to_30_seconds:
        ds = ds.filter(lambda example: example['audio_length'] <= 30)
        print(f"Filtered dataset from {orig_len} to {len(ds)} examples with audio length <= 30 seconds")
    return ds

In [13]:
# The following warning can be ignored:
# "The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results."
# See: https://discuss.huggingface.co/t/finetuning-whisper-attention-mask-not-set-and-canot-be-inferred/97456
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

## Download datasets and prepare features

--> no need to change anything here, just run

### Optimizing some settings for dataset access

In [14]:
datasets.disable_caching()
print('cache:', datasets.is_caching_enabled())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('device is: ', device)

# IMPORTANT! need to set to 1 to avoid the mapping to hang!
torch.set_num_threads(1)
torch.get_num_threads()

num_proc = min(32, os.cpu_count())
print('# processors:', num_proc)



cache: False
device is:  cuda
# processors: 24


### Load feature extractor

--> for the model type you specified above

In [15]:

# Load processor
print('Using Language: ', LANGUAGE)
print('Using model:', WHISPER_MODEL_TYPE)
processor = WhisperProcessor.from_pretrained(WHISPER_MODEL_TYPE, language=LANGUAGE, task=TASK)

# since this tokenizer isn't a FastTokenizer, so there is no point in running it with is_batched=True
# see: processor.tokenizer.is_fast
def prepare_features(example):
    example["input_features"] = processor.feature_extractor(example["audio"]["array"], sampling_rate=example["audio"]["sampling_rate"]).input_features[0]
    example["labels"] = processor.tokenizer(example["transcription"]).input_ids
    # also count number of tokens
    example["token_length"] = len(example["labels"])
    return example

Using Language:  en
Using model: openai/whisper-large-v3


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

### Load non-standard speech dataset

We need to filter to 30 seconds, as Whisper can only train on that.

In [16]:
train_dataset = load_dataset(DATASET_NAME, split='train', limit_to_30_seconds=True)
train_dataset = train_dataset.map(prepare_features, remove_columns=['audio'], writer_batch_size=1, num_proc=num_proc)
print(f"Loaded TRAIN dataset with {len(train_dataset)} examples")

Generating test split:   0%|          | 0/993 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/572 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/4236 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4236 [00:00<?, ? examples/s]

Filtered dataset from 4236 to 3130 examples with audio length <= 30 seconds


Map (num_proc=24):   0%|          | 0/3130 [00:00<?, ? examples/s]

Loaded TRAIN dataset with 3130 examples


In [17]:
test_dataset = load_dataset(DATASET_NAME, split='test', limit_to_30_seconds=True)
test_dataset = test_dataset.map(prepare_features, remove_columns=['audio'], writer_batch_size=1, num_proc=num_proc)
print(f"Loaded TEST dataset with {len(test_dataset)} examples")

Filter:   0%|          | 0/993 [00:00<?, ? examples/s]

Filtered dataset from 993 to 705 examples with audio length <= 30 seconds


Map (num_proc=24):   0%|          | 0/705 [00:00<?, ? examples/s]

Loaded TEST dataset with 705 examples


In [18]:
dev_dataset = load_dataset(DATASET_NAME, split='validation', limit_to_30_seconds=True)
dev_dataset = dev_dataset.map(prepare_features, remove_columns=['audio'], writer_batch_size=1, num_proc=num_proc)
print(f"Loaded DEV dataset with {len(dev_dataset)} examples")

Filter:   0%|          | 0/572 [00:00<?, ? examples/s]

Filtered dataset from 572 to 342 examples with audio length <= 30 seconds


Map (num_proc=24):   0%|          | 0/342 [00:00<?, ? examples/s]

Loaded DEV dataset with 342 examples


## Prepare Trainer

--> no need to change anything here, just run

Whenever something is changed in the settings, you need to rerun this part.

In [19]:
base_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL_NAME)
_ = base_model.to(device)
print('Using Language: ', LANGUAGE)
print('Using model:', WHISPER_MODEL_TYPE)

# ensure task and language for training
base_model.generation_config.language = LANGUAGE
base_model.generation_config.task = TASK
base_model.generation_config.forced_decoder_ids = None
base_model.config.forced_decoder_ids = None
# to use gradient checkpointing
base_model.config.use_cache = False
print('language set to:', base_model.generation_config.language)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

Using Language:  en
Using model: openai/whisper-large-v3
language set to: en


In [20]:
# Add SpecAugment
if USE_SPECAUGMENT:
    base_model.config.apply_spec_augment = USE_SPECAUGMENT

    # Specaugment (use default settings, as per paper)

    # time masking
    base_model.config.mask_time_prob = 0.05
    base_model.config.mask_time_length = 10
    base_model.config.mask_time_min_masks = 2

    # feature masking
    base_model.config.mask_feature_prob = 0.05 # def: 0
    base_model.config.mask_feature_length = 10
    base_model.config.mask_feature_min_masks = 2 # def: 0

print('Using specaugment:', base_model.config.apply_spec_augment)


Using specaugment: True


In [21]:
# which layers to tune

print("Updating encoder:", UPDATE_ENCODER)
print("Updating projection layer:", UPDATE_PROJ)
print("Updating decoder:", UPDATE_DECODER)


base_model.model.encoder.requires_grad_(UPDATE_ENCODER)
base_model.model.decoder.requires_grad_(UPDATE_DECODER)
base_model.proj_out.requires_grad_(UPDATE_PROJ)

print("Overview to number of model parameters to be updated:")
print('* encoder params to update/total:', count_trainable_parameters(base_model.model.encoder), base_model.model.encoder.num_parameters())
print('* decoder parans to update/total:', count_trainable_parameters(base_model.model.decoder), base_model.model.decoder.num_parameters())

print('* overall # trainable parameters:', count_trainable_parameters(base_model))
print('*     overall # model parameters:', base_model.model.num_parameters())

Updating encoder: True
Updating projection layer: True
Updating decoder: False
Overview to number of model parameters to be updated:
* encoder params to update/total: 636968960 636968960
* decoder parans to update/total: 66388480 906521600
* overall # trainable parameters: 703357440
*     overall # model parameters: 1543490560


In [22]:
# Training Hyper Parameters
# don't change settings here, but instead at very top!
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    logging_dir=os.path.join(OUTPUT_DIR, 'logs'),
    logging_steps=LOGGING_STEPS,
    report_to=["tensorboard"],
    include_num_input_tokens_seen=True,
    ### on GPU, can either do fp16 or bf16 depending on specific GPU
    fp16=USE_FP16, 
    bf16=USE_BF16, 
    push_to_hub=False,
    remove_unused_columns=False,
    #
    num_train_epochs=MAX_EPOCHS,
    max_steps=MAX_STEPS,
    #
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    #
    per_device_train_batch_size=BATCH_SIZE,
    #
    eval_on_start=EVAL_ON_START,
    predict_with_generate=True,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    eval_steps=EVAL_STEPS,
    eval_strategy="steps",
    generation_max_length=MAX_GEN_LEN,
    #
    metric_for_best_model="wer",
    greater_is_better=False,
    #
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    #
    # only applies to polynomial schedule (constant ignores args)
    lr_scheduler_kwargs={
        "lr_end": LR_END, # The final LR.  Crucial for polynomial decay.
        "power": LR_DECAY_POWER, # for decay
        # we don't need to set the other arguments as they are already set in the args outside
        #"num_warmup_steps": WARMUP_STEPS, # The number of steps for the warmup phase.
        #"num_training_steps": MAX_STEPS, # The total number of training steps.
        #"lr_init": 1e-5 # we take the LR setting
    },

    learning_rate=LEARNING_RATE,
    warmup_steps=LR_WARMUP_STEPS, # what happens if we have this and the LR schedule args ?
    #
    save_steps=SAVE_STEPS,
    save_strategy="steps",
    save_total_limit=NUM_CHECKPOINTS_TO_STORE,
    load_best_model_at_end=True,
    # group_by_length=True
    # auto_find_batch_size=True
)

print('trainer args set, writing to:', OUTPUT_DIR)

trainer args set, writing to: /jupyter_kernel/trained_models/en_nonstandard_tune_whisper_large_v3_baseline


In [23]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=base_model.config.decoder_start_token_id,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=base_model,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor
)


Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


## Run the training

Note: tensorboard doesn't show properly in jupyter notebooks, use the tensorboard_server.py tool to host a tensorboard instance on Modal, using below model training dir:

In [24]:
print('model training dir:', OUTPUT_DIR)

model training dir: /jupyter_kernel/trained_models/en_nonstandard_tune_whisper_large_v3_baseline


In [25]:
# train from scratch
# trainer.train()

# # alternatively, you can continue training if a previous job was interrupted
trainer.train(resume_from_checkpoint = True)


There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Step,Training Loss,Validation Loss,Wer,Cer,Input Tokens Seen
200,0.670600,0.588253,0.166670,0.098805,614400000
300,0.658700,0.557633,0.155871,0.092836,921600000
400,0.484700,0.541017,0.153737,0.091750,1226496000
500,0.433800,0.540500,0.131215,0.071058,1533696000
600,0.374300,0.533526,0.130914,0.072273,1840896000
700,0.351900,0.531478,0.132066,0.071950,2148096000
800,0.342600,0.526069,0.129884,0.070896,2452992000
900,0.289500,0.526272,0.131130,0.072035,2760192000
1000,0.319800,0.526469,0.131084,0.071916,3067392000


adjusted: 0.16667016762804204 0.09880476775551605
un-adjusted: 0.21591790167647265 0.11456197183867795
adjusted: 0.15587106105998375 0.09283574356796381
un-adjusted: 0.17266217077872506 0.09933346936393522


/usr/local/lib/python3.11/site-packages/transformers/modeling_utils.py:3353: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


adjusted: 0.15373666661383173 0.09174975479155752
un-adjusted: 0.2198079529961168 0.12064454357626722
adjusted: 0.1312145898824608 0.07105842037265576
un-adjusted: 0.13229723652431796 0.07105842037265576
adjusted: 0.13091427090361568 0.07227325230489129
un-adjusted: 0.13199691754547282 0.07227325230489129
adjusted: 0.13206617533561274 0.0719497682994844
un-adjusted: 0.13314882197746988 0.0719497682994844
adjusted: 0.12988438942825686 0.07089636978678163
un-adjusted: 0.13149866818069353 0.07089636978678163
adjusted: 0.13112992552532876 0.07203544969280173
un-adjusted: 0.13274420427776543 0.07203544969280173
adjusted: 0.13108424379076766 0.07191554962638053
un-adjusted: 0.13269852254320433 0.07191554962638053


There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=1000, training_loss=0.3527598714828491, metrics={'train_runtime': 8599.8547, 'train_samples_per_second': 0.93, 'train_steps_per_second': 0.116, 'total_flos': 2.713921647280128e+19, 'train_loss': 0.3527598714828491, 'epoch': 2.5510204081632653, 'num_input_tokens_seen': 3067392000})

### On DEV set

In [26]:
# (should give the same result shown in trainig progress on dev set)
trainer.evaluate(dev_dataset, language=LANGUAGE)

adjusted: 0.12988438942825686 0.07089636978678163
un-adjusted: 0.13149866818069353 0.07089636978678163


{'eval_loss': 0.5260688066482544,
 'eval_wer': 0.12988438942825686,
 'eval_cer': 0.07089636978678163,
 'eval_runtime': 467.372,
 'eval_samples_per_second': 0.732,
 'eval_steps_per_second': 0.092,
 'epoch': 2.5510204081632653,
 'num_input_tokens_seen': 3067392000}

### On TEST set

In [28]:
# run on dev-set 
# (should give the same result shown in trainig progress on dev set)
trainer.evaluate(test_dataset, language=LANGUAGE)

adjusted: 0.08850797769915196 0.04493261913876007
un-adjusted: 0.08860254034690608 0.04493261913876007


{'eval_loss': 0.5018638968467712,
 'eval_wer': 0.08850797769915196,
 'eval_cer': 0.04493261913876007,
 'eval_runtime': 982.2196,
 'eval_samples_per_second': 0.718,
 'eval_steps_per_second': 0.091,
 'epoch': 2.5510204081632653,
 'num_input_tokens_seen': 3067392000}

## Store Model

--> save best model

### Save to your volume

In [29]:
# with "load_best_model_at_end=True" set in the settings (this is the default, so don't change that), after training is completed the best model is loaded and then saved
best_model_dir = os.path.join(OUTPUT_DIR, 'best_model')
print(f"Saving to: {best_model_dir}")
trainer.model.save_pretrained(best_model_dir, safe_serialization=True)
trainer.tokenizer.save_pretrained(best_model_dir)

Saving to: /jupyter_kernel/trained_models/en_nonstandard_tune_whisper_large_v3_baseline/best_model


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


[]

### Pushing to Hugging Face 🤗

- Modify for each model that I'm pushing to hugging face 🤗

In [30]:
trainer.model.push_to_hub("smainye/eng_finetunned_tune_whisper_large_v3_model_baseline")
trainer.tokenizer.push_to_hub("smainye/eng_finetunned_tune_whisper_large_v3_model_baseline")

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.18G [00:00<?, ?B/s]

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/smainye/eng_finetunned_tune_whisper_large_v3_model_baseline/commit/49e23e6082c082788758cf69d11f5c4922f8dc68', commit_message='Upload processor', commit_description='', oid='49e23e6082c082788758cf69d11f5c4922f8dc68', pr_url=None, repo_url=RepoUrl('https://huggingface.co/smainye/eng_finetunned_tune_whisper_large_v3_model_baseline', endpoint='https://huggingface.co', repo_type='model', repo_id='smainye/eng_finetunned_tune_whisper_large_v3_model_baseline'), pr_revision=None, pr_num=None)